In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

path = '/content/drive/MyDrive/ESAA data/'   # ← 복사한 경로에서 파일명만 뺀 부분으로 바꾸기

train = pd.read_csv(path + 'movies_train.csv')
test = pd.read_csv(path + 'movies_test.csv')
submission = pd.read_csv(path + 'submission.csv')

print(train.shape, test.shape, submission.shape)

(600, 12) (243, 11) (243, 2)


In [3]:
def build_X():
    df = pd.concat([train.drop(columns='box_off_num'), test], ignore_index=True)

    top5 = df['distributor'].iloc[:len(train)].value_counts().index[:5]
    df['distributor'] = df['distributor'].where(df['distributor'].isin(top5), '기타')

    df = df.drop(columns='dir_prev_bfnum')

    df['년'] = df['release_time'].str[:4].astype(int)
    df['월'] = df['release_time'].str[5:7].astype(int)

    df = df.drop(columns=['title', 'director', 'release_time'])
    df = pd.get_dummies(df)
    return df.iloc[:len(train)], df.iloc[len(train):]

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def run(name, model):
    X, X_test = build_X()
    y = train['box_off_num']

    # 교차검증 점수
    scores = []
    for tr, va in kf.split(X):
        model.fit(X.iloc[tr], y.iloc[tr])
        pred = model.predict(X.iloc[va])
        scores.append(np.sqrt(mean_squared_error(y.iloc[va], pred)))
    print(f'{name} CV RMSE = {np.mean(scores):,.0f}')

    # 전체 학습 후 제출 파일 저장
    model.fit(X, y)
    sub = submission.copy()
    sub['box_off_num'] = model.predict(X_test)
    sub.to_csv(path + name + '.csv', index=False)

In [5]:
run('1_RF', RandomForestRegressor(n_estimators=300, random_state=42))

1_RF CV RMSE = 1,495,659


In [6]:
run('2_ET', ExtraTreesRegressor(n_estimators=300, random_state=42))

2_ET CV RMSE = 1,482,065


단계	CV RMSE

1단계_RF	1,495,659

2단계_ET	1,482,065 ✅

3단계: dir_prev_bfnum 결측치를 0으로 채우기

ET가 약 1만 4천 낮아서 조금 더 좋아요. 다만 차이가 1% 정도라 거의 비슷한 수준이에요. 그래서 3단계도 RF와 ET를 둘 다 돌려 볼게요.

In [7]:
def build_X():
    df = pd.concat([train.drop(columns='box_off_num'), test], ignore_index=True)

    top5 = df['distributor'].iloc[:len(train)].value_counts().index[:5]
    df['distributor'] = df['distributor'].where(df['distributor'].isin(top5), '기타')

    df['dir_prev_bfnum'] = df['dir_prev_bfnum'].fillna(0)   # ★ 삭제 대신 0으로 채움

    df['년'] = df['release_time'].str[:4].astype(int)
    df['월'] = df['release_time'].str[5:7].astype(int)

    df = df.drop(columns=['title', 'director', 'release_time'])
    df = pd.get_dummies(df)
    return df.iloc[:len(train)], df.iloc[len(train):]

In [8]:
run('3_결측0_RF', RandomForestRegressor(n_estimators=300, random_state=42))
run('3_결측0_ET', ExtraTreesRegressor(n_estimators=300, random_state=42))

3_결측0_RF CV RMSE = 1,481,411
3_결측0_ET CV RMSE = 1,489,259


3단계 결과

모델	2단계 → 3단계	변화

RF	1,495,659 → 1,481,411	▼ 1.4만 좋아짐 ✅

ET	1,482,065 → 1,489,259	▲ 0.7만 나빠짐

4단계: 타깃 √ 변환

In [9]:
def run(name, model):
    X, X_test = build_X()
    y = train['box_off_num']
    y_t = np.sqrt(y)                                    # ★ 타깃에 √ 씌움

    # 교차검증 점수
    scores = []
    for tr, va in kf.split(X):
        model.fit(X.iloc[tr], y_t.iloc[tr])
        pred = np.clip(model.predict(X.iloc[va]), 0, None) ** 2   # ★ 제곱해서 원래 단위로
        scores.append(np.sqrt(mean_squared_error(y.iloc[va], pred)))
    print(f'{name} CV RMSE = {np.mean(scores):,.0f}')

    # 전체 학습 후 제출 파일 저장
    model.fit(X, y_t)
    sub = submission.copy()
    sub['box_off_num'] = np.clip(model.predict(X_test), 0, None) ** 2   # ★
    sub.to_csv(path + name + '.csv', index=False)

In [10]:
run('4_sqrt_RF', RandomForestRegressor(n_estimators=300, random_state=42))
run('4_sqrt_ET', ExtraTreesRegressor(n_estimators=300, random_state=42))

4_sqrt_RF CV RMSE = 1,498,395
4_sqrt_ET CV RMSE = 1,445,515


4단계 결과

모델	3단계 → 4단계	변화

RF	1,481,411 → 1,498,395	▲ 1.7만 나빠짐

ET	1,489,259 → 1,445,515	▼ 4.4만 좋아짐 ✅ 지금까지 최고

5단계: 배급사 이름 정리

(주)쇼박스와 쇼박스(주)미디어플렉스처럼 같은 회사가 다른 이름으로 적힌 걸 하나로 합쳐요.

In [11]:
def clean_name(name):
    name = name.replace('(주)', '').replace('㈜', '').replace('주식회사', '').replace(' ', '')
    for key in ['CJ', '쇼박스', '롯데', 'NEW', '마운틴픽쳐스']:
        if key in name:
            return key
    return name

def build_X():
    df = pd.concat([train.drop(columns='box_off_num'), test], ignore_index=True)

    df['distributor'] = df['distributor'].apply(clean_name)   # ★ 배급사 이름 정리

    top5 = df['distributor'].iloc[:len(train)].value_counts().index[:5]
    df['distributor'] = df['distributor'].where(df['distributor'].isin(top5), '기타')

    df['dir_prev_bfnum'] = df['dir_prev_bfnum'].fillna(0)

    df['년'] = df['release_time'].str[:4].astype(int)
    df['월'] = df['release_time'].str[5:7].astype(int)

    df = df.drop(columns=['title', 'director', 'release_time'])
    df = pd.get_dummies(df)
    return df.iloc[:len(train)], df.iloc[len(train):]

In [12]:
run('5_배급사_RF', RandomForestRegressor(n_estimators=300, random_state=42))
run('5_배급사_ET', ExtraTreesRegressor(n_estimators=300, random_state=42))

5_배급사_RF CV RMSE = 1,495,898
5_배급사_ET CV RMSE = 1,444,949


5단계 결과

모델	4단계 → 5단계	변화

RF	1,498,395 → 1,495,898	▼ 0.25만 (미미)

ET	1,445,515 → 1,444,949	▼ 0.06만 (미미) ✅ 최고 기록 갱신

거의 변화가 없어요. 배급사를 상위 5개와 '기타'로만 나누기 때문에, 이름을 정리해도 그룹이 크게 바뀌지 않았을 가능성이 커요. 그래도 조금이나마 둘 다 좋아졌고 손해는 없으니 유지해요.

6단계: RF + ET 앙상블

지금까지 보면 RF는 √ 없이, ET는 √ 있을 때 더 좋았어요. 그래서 두 가지를 비교해요.

A: RF(√) + ET(√)

B: RF(원래 그대로) + ET(√) ← 각자 잘 맞는 방식으로

In [13]:
def fit_pred(model, use_sqrt, X_tr, y_tr, X_pred):
    y_t = np.sqrt(y_tr) if use_sqrt else y_tr
    model.fit(X_tr, y_t)
    p = model.predict(X_pred)
    return np.clip(p, 0, None) ** 2 if use_sqrt else p

def run_ens(name, models):
    X, X_test = build_X()
    y = train['box_off_num']

    scores = []
    for tr, va in kf.split(X):
        preds = [fit_pred(m, s, X.iloc[tr], y.iloc[tr], X.iloc[va]) for m, s in models]
        scores.append(np.sqrt(mean_squared_error(y.iloc[va], np.mean(preds, axis=0))))
    print(f'{name} CV RMSE = {np.mean(scores):,.0f}')

    preds = [fit_pred(m, s, X, y, X_test) for m, s in models]
    sub = submission.copy()
    sub['box_off_num'] = np.mean(preds, axis=0)
    sub.to_csv(path + name + '.csv', index=False)

In [14]:
rf = RandomForestRegressor(n_estimators=300, random_state=42)
et = ExtraTreesRegressor(n_estimators=300, random_state=42)

run_ens('6A_RF√+ET√', [(rf, True),  (et, True)])
run_ens('6B_RF+ET√',  [(rf, False), (et, True)])
# (True = √ 변환 사용, False = 사용 안 함)

6A_RF√+ET√ CV RMSE = 1,448,776
6B_RF+ET√ CV RMSE = 1,428,288


6단계 결과

조합	CV RMSE	5단계 ET 대비

6A: RF(√) + ET(√)	1,448,776	▲ 0.4만 나빠짐

6B: RF(원래) + ET(√)	1,428,288	▼ 1.7만 좋아짐 ✅ 최고 기록

6B가 지금까지 가장 좋아요. 각 모델을 잘 맞는 방식으로 학습시킨 뒤 평균 내니 효과가 있었어요. 반면 6A처럼 둘 다 √로 맞추면 RF가 발목을 잡았어요.

6B_RF+ET√.csv를 제출하세요.

In [15]:
from google.colab import files
files.download(path + '6B_RF+ET√.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1. 앙상블 비율 조절 ⭐ 가장 쉬움

지금은 RF와 ET를 5:5로 평균 내고 있어요. ET가 더 강하니까 3:7이나 2:8이 더 나을 수 있어요. 코드 몇 줄이면 돼요.

In [17]:
X, X_test = build_X()
y = train['box_off_num']

# 교차검증 예측을 한 번만 만들어 두고
oof_rf = np.zeros(len(y))
oof_et = np.zeros(len(y))
for tr, va in kf.split(X):
    oof_rf[va] = fit_pred(rf, False, X.iloc[tr], y.iloc[tr], X.iloc[va])
    oof_et[va] = fit_pred(et, True,  X.iloc[tr], y.iloc[tr], X.iloc[va])

# 비율만 바꿔가며 점수 비교
for w in [0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
    pred = w * oof_rf + (1 - w) * oof_et
    print(f'RF {w:.0%} : ET {1-w:.0%}  →  RMSE {np.sqrt(mean_squared_error(y, pred)):,.0f}')

RF 0% : ET 100%  →  RMSE 1,475,547
RF 20% : ET 80%  →  RMSE 1,460,083
RF 30% : ET 70%  →  RMSE 1,456,049
RF 40% : ET 60%  →  RMSE 1,454,512
RF 50% : ET 50%  →  RMSE 1,455,480
RF 60% : ET 40%  →  RMSE 1,458,947
RF 80% : ET 20%  →  RMSE 1,473,299
RF 100% : ET 0%  →  RMSE 1,497,282


2번: 하이퍼파라미터 튜닝

RF는 √ 없이, ET는 √로 학습하는 조건을 지키기 위해 GridSearchCV 대신 직접 반복문으로 비교해요. (GridSearchCV는 √ 공간에서 채점해서 실제 관객수 기준 점수가 안 나와요.)

In [18]:
def cv_score(model, use_sqrt):
    X, _ = build_X()
    y = train['box_off_num']
    scores = []
    for tr, va in kf.split(X):
        p = fit_pred(model, use_sqrt, X.iloc[tr], y.iloc[tr], X.iloc[va])
        scores.append(np.sqrt(mean_squared_error(y.iloc[va], p)))
    return np.mean(scores)

In [19]:
for leaf in [1, 3, 5, 10]:
    for feat in [1.0, 0.7, 0.5]:
        rf_s = cv_score(RandomForestRegressor(n_estimators=300, min_samples_leaf=leaf,
                        max_features=feat, random_state=42, n_jobs=-1), False)
        et_s = cv_score(ExtraTreesRegressor(n_estimators=300, min_samples_leaf=leaf,
                        max_features=feat, random_state=42, n_jobs=-1), True)
        print(f'leaf={leaf:<2} feat={feat}  |  RF {rf_s:,.0f}  |  ET {et_s:,.0f}')

leaf=1  feat=1.0  |  RF 1,475,696  |  ET 1,444,949
leaf=1  feat=0.7  |  RF 1,454,850  |  ET 1,402,559
leaf=1  feat=0.5  |  RF 1,433,080  |  ET 1,419,494
leaf=3  feat=1.0  |  RF 1,475,181  |  ET 1,407,023
leaf=3  feat=0.7  |  RF 1,445,769  |  ET 1,413,567
leaf=3  feat=0.5  |  RF 1,415,668  |  ET 1,424,712
leaf=5  feat=1.0  |  RF 1,460,376  |  ET 1,408,394
leaf=5  feat=0.7  |  RF 1,436,435  |  ET 1,417,682
leaf=5  feat=0.5  |  RF 1,420,550  |  ET 1,435,973
leaf=10 feat=1.0  |  RF 1,457,404  |  ET 1,432,650
leaf=10 feat=0.7  |  RF 1,434,388  |  ET 1,444,951
leaf=10 feat=0.5  |  RF 1,416,086  |  ET 1,469,936


비교 기준 (현재 기본 설정 leaf=1, feat=1.0): RF 1,475,696(표의 첫줄) / ET 1,444,949

7단계: 튜닝된 모델로 앙상블 다시 만들기

6단계에서 만든 run_ens 함수를 그대로 써요.

비교 기준: 6B 1,428,288 / 튜닝된 ET 단독 1,402,559

In [20]:
rf_best = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.5,
                                random_state=42, n_jobs=-1)
et_best = ExtraTreesRegressor(n_estimators=300, min_samples_leaf=1, max_features=0.7,
                              random_state=42, n_jobs=-1)

run_ens('7_튜닝앙상블', [(rf_best, False), (et_best, True)])

7_튜닝앙상블 CV RMSE = 1,384,547


7단계 결과

비교 대상	CV RMSE	7단계 대비

1단계 베이스라인 RF	1,495,659	▼ 11.1만 (7.4%) 개선

6B 앙상블 (튜닝 전)	1,428,288	▼ 4.4만

튜닝된 ET 단독	1,402,559	▼ 1.8만

7단계 튜닝 앙상블	1,384,547	최고 기록

튜닝과 앙상블이 각각 효과를 내고, 합쳤을 때도 효과가 겹쳤어요.

📌 7_튜닝앙상블.csv를 제출하세요. 6B 리더보드 점수(1,295,372)보다 좋아지는지 확인해 보세요.

In [21]:
from google.colab import files
files.download(path + '7_튜닝앙상블.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>